# 04 -- DIAL Generation Demo

Apply the DIAL transform (Dialectal Interpolation via Algebraic Linearisation)
at various alpha intensities and observe how generated output changes.

**Key formula:** W(alpha) = P @ Lambda^alpha @ P^{-1}

**Key modules:** `generative.dial`, `generative.intensity`, `generative.mixing`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from eigendialectos.constants import DialectCode, DIALECT_NAMES
from eigendialectos.types import EmbeddingMatrix, EigenDecomposition
from eigendialectos.spectral.transformation import compute_transformation_matrix
from eigendialectos.spectral.eigendecomposition import eigendecompose
from eigendialectos.generative.dial import (
    apply_dial, dial_transform_embedding, compute_dial_series,
)

## Setup: Build Eigendecomposition

In [ ]:
rng = np.random.default_rng(42)
dim, vocab_size = 30, 100
vocab = [f"w{i}" for i in range(vocab_size)]

base = rng.standard_normal((dim, vocab_size)).astype(np.float64)
target = base + rng.standard_normal((dim, vocab_size)) * 0.2

ref_emb = EmbeddingMatrix(data=base, vocab=vocab, dialect_code=DialectCode.ES_PEN)
tgt_emb = EmbeddingMatrix(data=target, vocab=vocab, dialect_code=DialectCode.ES_RIO)

W = compute_transformation_matrix(ref_emb, tgt_emb, method="lstsq", regularization=0.01)
eigen = eigendecompose(W)

print(f"Transformation shape: {W.shape}")
print(f"Eigendecomposition rank: {eigen.rank}")
print(f"Top 5 eigenvalue magnitudes: {np.sort(np.abs(eigen.eigenvalues))[::-1][:5].real.round(4)}")

## DIAL Transform at Various Alpha Values

- alpha=0 -> identity (neutral)
- alpha=1 -> full dialect
- alpha>1 -> hyperdialect

In [ ]:
alphas = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]

print(f"{'Alpha':>8}  {'||W(a)||_F':>12}  {'||W(a)-I||_F':>14}  {'cond(W(a))':>12}")
print("-" * 52)
for alpha in alphas:
    W_alpha = apply_dial(eigen, alpha)
    norm_f = np.linalg.norm(W_alpha.data, "fro")
    dist_I = np.linalg.norm(W_alpha.data - np.eye(dim), "fro")
    cond = np.linalg.cond(W_alpha.data)
    print(f"{alpha:>8.2f}  {norm_f:>12.4f}  {dist_I:>14.4f}  {cond:>12.1f}")

## Embedding Transformation

In [ ]:
# Create sample input embeddings
n_samples = 20
input_embs = rng.standard_normal((n_samples, dim)).astype(np.float64)

results = []
for alpha in np.arange(0.0, 1.6, 0.1):
    transformed = dial_transform_embedding(input_embs, eigen, alpha)
    delta = transformed - input_embs
    mean_displacement = float(np.mean(np.linalg.norm(delta, axis=1)))
    mean_norm = float(np.mean(np.linalg.norm(transformed, axis=1)))
    results.append((alpha, mean_displacement, mean_norm))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot([r[0] for r in results], [r[1] for r in results], "o-", color="steelblue")
ax1.set_xlabel("alpha")
ax1.set_ylabel("Mean displacement from neutral")
ax1.set_title("Embedding Displacement vs Alpha")

ax2.plot([r[0] for r in results], [r[2] for r in results], "s-", color="teal")
ax2.set_xlabel("alpha")
ax2.set_ylabel("Mean embedding norm")
ax2.set_title("Embedding Norm vs Alpha")

plt.tight_layout()
plt.show()

## DIAL Series: Full Sweep

In [ ]:
series = compute_dial_series(eigen, alpha_range=(0.0, 1.5, 0.1))
print(f"Generated {len(series)} DIAL transforms")

# Plot Frobenius norm of W(alpha) - W(0) over the series
norms = [float(np.linalg.norm(W_a.data - np.eye(dim), 'fro')) for W_a in series]
alphas_plot = np.arange(0.0, 1.5, 0.1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(alphas_plot, norms, "o-", color="crimson")
ax.set_xlabel("alpha")
ax.set_ylabel("||W(alpha) - I||_F")
ax.set_title("DIAL Transform Distance from Identity")
plt.tight_layout()
plt.show()

## Dialect Mixing Demo

In [ ]:
from eigendialectos.generative.mixing import mix_dialects

# Build a second eigendecomposition for mixing
target2 = base + rng.standard_normal((dim, vocab_size)) * 0.25
tgt2_emb = EmbeddingMatrix(data=target2, vocab=vocab, dialect_code=DialectCode.ES_MEX)
W2 = compute_transformation_matrix(ref_emb, tgt2_emb, method="lstsq", regularization=0.01)

# Mix at various weights
for beta in [0.0, 0.25, 0.5, 0.75, 1.0]:
    W_mix = mix_dialects([(W, 1.0 - beta), (W2, beta)])
    print(
        f"  beta(MEX)={beta:.2f}: ||W_mix||_F = {np.linalg.norm(W_mix.data, 'fro'):.4f}, "
        f"cond = {np.linalg.cond(W_mix.data):.1f}"
    )